# Übung 4: Plotting und Visualisierung für die Energiewirtschaft

**Ziel:** Erstellen von Diagrammen mit Python. Pro Diagrammtyp gibt es eine Matplotlib- (statisch) und eine Plotly-Variante (interaktiv).

## 4.1 Setup und Daten laden

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.express as px
import plotly.io as pio
from plotly.subplots import make_subplots

# Saubere Defaults — einmal global, danach Plot-Code minimal halten
pio.templates.default = "plotly_white"
plt.rcParams["figure.figsize"] = (12, 4)
plt.rcParams["axes.grid"]      = True
plt.rcParams["grid.alpha"]     = 0.3
plt.rcParams["axes.axisbelow"] = True   # Grid hinter Daten

# Plotly: dezentes Grid + Null-Linie standardmäßig sichtbar
pio.templates["plotly_white"].layout.xaxis.update(zeroline=True, zerolinecolor="#888", zerolinewidth=1)
pio.templates["plotly_white"].layout.yaxis.update(zeroline=True, zerolinecolor="#888", zerolinewidth=1)

# Daten aus der Zeitreihen-Übung laden (oder synthetisch erzeugen)
try:
    df_last  = pd.read_csv("../3_Zeitreihen/haushaltslast_2024.csv", index_col=0, parse_dates=True)
    df_preis = pd.read_csv("../3_Zeitreihen/strompreise_2024.csv",   index_col=0, parse_dates=True)
except FileNotFoundError:
    np.random.seed(42)
    idx = pd.date_range("2024-01-01", "2024-12-31 23:00", freq="h")
    n = len(idx); h = idx.hour; m = idx.month
    last_kw = (0.3 + 0.5*np.sin((h-6)*np.pi/12).clip(0)
                   + 0.4*np.sin((h-17)*np.pi/6).clip(0)
                   + np.random.normal(0, 0.15, n)).clip(0.1)
    preise = (70 + 20*np.sin((m-1)*2*np.pi/12) + 15*np.sin((h-6)*np.pi/12)
              + np.random.normal(0, 12, n))
    preise[np.random.choice(n, 150, replace=False)] = np.random.uniform(-20, -1, 150)
    df_last  = pd.DataFrame({"last_kw":       last_kw}, index=idx)
    df_preis = pd.DataFrame({"price_eur_mwh": preise},  index=idx)

df = df_last.join(df_preis)
df["kosten_eur"] = df["last_kw"] * df["price_eur_mwh"] / 1000
monate = ["Jan","Feb","Mär","Apr","Mai","Jun",
          "Jul","Aug","Sep","Okt","Nov","Dez"]
df.head(3)


## 4.2 Linienplot — Wochenansicht der Last

**Matplotlib (statisch)**

In [ ]:
woche = df.loc["2024-06-10":"2024-06-16", "last_kw"]

woche.plot(title="Haushaltslast — Beispielwoche Juni 2024",
           ylabel="Leistung (kW)")
plt.axhline(woche.mean(), color="red", linestyle="--",
            label=f"Mittelwert {woche.mean():.2f} kW")
plt.ylim(bottom=0)              # Null-Linie sichtbar (Konvention bei Lastdaten)
plt.legend()
plt.show()


**Plotly (interaktiv)**

In [ ]:
woche = df.loc["2024-06-10":"2024-06-16", "last_kw"]

fig = px.line(woche, title="Haushaltslast — Beispielwoche Juni 2024",
              labels={"value": "Leistung (kW)", "timestamp": ""})
fig.add_hline(y=woche.mean(), line_dash="dash", line_color="red",
              annotation_text=f"Mittelwert {woche.mean():.2f} kW")
fig.update_yaxes(rangemode="tozero")
fig.show()


## 4.3 Linienplot mit zwei Y-Achsen — Tagesprofil

**Matplotlib (statisch)**

In [ ]:
tag = df.loc["2024-07-15"]

fig, ax1 = plt.subplots()
ax1.plot(tag.index.hour, tag["last_kw"], color="C0", label="Last")
ax2 = ax1.twinx()
ax2.plot(tag.index.hour, tag["price_eur_mwh"], color="C3",
         linestyle="--", label="Preis")

ax1.set_xlabel("Stunde")
ax1.set_ylabel("Last (kW)",       color="C0")
ax2.set_ylabel("Preis (EUR/MWh)", color="C3")
ax1.set_ylim(bottom=0)
ax2.axhline(0, color="C3", linewidth=0.5, alpha=0.5)
plt.title("Tagesprofil — 15.07.2024")
plt.show()


**Plotly (interaktiv)**

In [ ]:
tag = df.loc["2024-07-15"]

fig = make_subplots(specs=[[{"secondary_y": True}]])
fig.add_trace(go.Scatter(x=tag.index.hour, y=tag["last_kw"], name="Last (kW)"),
              secondary_y=False)
fig.add_trace(go.Scatter(x=tag.index.hour, y=tag["price_eur_mwh"],
                         name="Preis (EUR/MWh)", line=dict(dash="dash")),
              secondary_y=True)
fig.update_layout(title="Tagesprofil — 15.07.2024", xaxis_title="Stunde")
fig.update_yaxes(title_text="Last (kW)",       secondary_y=False, rangemode="tozero")
fig.update_yaxes(title_text="Preis (EUR/MWh)", secondary_y=True)
fig.show()


## 4.4 Balkendiagramm — Monatlicher Verbrauch

**Matplotlib (statisch)**

In [ ]:
df_monat = df.resample("ME")["last_kw"].sum()
df_monat.index = monate

df_monat.plot(kind="bar", rot=0,
              title="Monatlicher Stromverbrauch 2024",
              ylabel="Verbrauch (kWh)")
plt.axhline(df_monat.mean(), color="gray", linestyle="--",
            label=f"Ø {df_monat.mean():.0f} kWh")
plt.legend()
plt.show()

**Plotly (interaktiv)**

In [ ]:
df_monat = df.resample("ME")["last_kw"].sum()

fig = px.bar(x=monate, y=df_monat.values,
             title="Monatlicher Stromverbrauch 2024",
             labels={"x": "Monat", "y": "Verbrauch (kWh)"})
fig.add_hline(y=df_monat.mean(), line_dash="dash", line_color="gray",
              annotation_text=f"Ø {df_monat.mean():.0f} kWh")
fig.show()

## 4.5 Gestapeltes Balkendiagramm — Kostenkomponenten

**Matplotlib (statisch)**

In [ ]:
df["kosten_arbeit_eur"] = df["last_kw"] * df["price_eur_mwh"].clip(lower=0) / 1000

df_monat_kosten = pd.DataFrame({
    "Arbeitskosten":   df.resample("ME")["kosten_arbeit_eur"].sum().values,
    "Leistungskosten": df.resample("ME")["last_kw"].max().values * 10,  # 10 €/kW/Monat
}, index=monate)

df_monat_kosten.plot(kind="bar", stacked=True, rot=0,
                      title="Monatliche Stromkosten — Arbeit + Leistung",
                      ylabel="Kosten (EUR)")
plt.show()

**Plotly (interaktiv)**

In [ ]:
fig = go.Figure()
fig.add_bar(x=monate, y=df_monat_kosten["Arbeitskosten"],   name="Arbeitskosten")
fig.add_bar(x=monate, y=df_monat_kosten["Leistungskosten"], name="Leistungskosten")
fig.update_layout(barmode="stack",
                  title="Monatliche Stromkosten — Arbeit + Leistung",
                  yaxis_title="Kosten (EUR)")
fig.show()

## 4.6 Histogramm — Preisverteilung

**Matplotlib (statisch)**

In [ ]:
df["price_eur_mwh"].plot(kind="hist", bins=50,
    title="Verteilung der Strompreise 2024")
plt.axvline(0, color="red", linestyle="--", label="0 EUR/MWh")
plt.axvline(df["price_eur_mwh"].mean(), color="orange", linestyle="--",
            label=f"Ø {df['price_eur_mwh'].mean():.1f}")
plt.xlabel("Preis (EUR/MWh)")
plt.legend()
plt.show()

**Plotly (interaktiv)**

In [ ]:
fig = px.histogram(df, x="price_eur_mwh", nbins=50,
                   title="Verteilung der Strompreise 2024",
                   labels={"price_eur_mwh": "Preis (EUR/MWh)"})
fig.add_vline(x=0, line_dash="dash", line_color="red",
              annotation_text="0 EUR/MWh")
fig.add_vline(x=df["price_eur_mwh"].mean(), line_dash="dash",
              line_color="orange",
              annotation_text=f"Ø {df['price_eur_mwh'].mean():.1f}")
fig.show()

## 4.7 Boxplot — Preisverteilung nach Monat

**Matplotlib (statisch)**

In [ ]:
df_box = df.copy()
df_box["monat"] = df_box.index.month

ax = df_box.boxplot(column="price_eur_mwh", by="monat")
ax.axhline(0, color="red", linestyle="--", linewidth=1, label="0 EUR/MWh")
ax.set_xticklabels(monate)
plt.suptitle("")  # automatischer "Boxplot grouped by..." entfernen
plt.title("Preisverteilung nach Monat")
plt.xlabel("Monat"); plt.ylabel("Preis (EUR/MWh)")
plt.legend()
plt.show()

**Plotly (interaktiv)**

In [ ]:
fig = px.box(df.assign(monat=df.index.month),
             x="monat", y="price_eur_mwh",
             title="Preisverteilung nach Monat",
             labels={"monat": "Monat", "price_eur_mwh": "Preis (EUR/MWh)"})
fig.add_hline(y=0, line_dash="dash", line_color="red",
              annotation_text="0 EUR/MWh")
fig.update_xaxes(tickmode="array", tickvals=list(range(1, 13)), ticktext=monate)
fig.show()

## 4.8 Heatmap — Stündliches Lastprofil

**Matplotlib (statisch)**

In [ ]:
heatmap = df["last_kw"].groupby(
    [df.index.date, df.index.hour]).mean().unstack()

plt.figure(figsize=(12, 6))
plt.imshow(heatmap.values, aspect="auto", cmap="YlOrRd")
plt.colorbar(label="Last (kW)")
plt.title("Heatmap — Haushaltslast 2024")
plt.xlabel("Stunde"); plt.ylabel("Tag des Jahres")
plt.show()

**Plotly (interaktiv)**

In [ ]:
fig = px.imshow(heatmap, aspect="auto", color_continuous_scale="YlOrRd",
                title="Heatmap — Haushaltslast 2024",
                labels=dict(x="Stunde", y="Tag", color="Last (kW)"))
fig.show()

## 4.9 Dauerlinie — Jahresdauerlinie

**Matplotlib (statisch)**

In [ ]:
last_sorted = df["last_kw"].sort_values(ascending=False).reset_index(drop=True)

last_sorted.plot(title="Jahresdauerlinie — Haushaltslast 2024",
                 xlabel="Stunden (sortiert)", ylabel="Leistung (kW)")
plt.axhline(df["last_kw"].quantile(0.9), color="red", linestyle="--",
            label=f"90%-Quantil  {df['last_kw'].quantile(0.9):.2f} kW")
plt.ylim(bottom=0)
plt.xlim(left=0)
plt.legend()
plt.show()


**Plotly (interaktiv)**

In [ ]:
last_sorted = df["last_kw"].sort_values(ascending=False).reset_index(drop=True)

fig = px.line(last_sorted, title="Jahresdauerlinie — Haushaltslast 2024",
              labels={"index": "Stunden (sortiert)", "value": "Leistung (kW)"})
fig.add_hline(y=df["last_kw"].quantile(0.9), line_dash="dash",
              line_color="red",
              annotation_text=f"90%-Quantil {df['last_kw'].quantile(0.9):.2f} kW")
fig.update_yaxes(rangemode="tozero")
fig.update_xaxes(rangemode="tozero")
fig.show()


## 4.10 Tagesprofil mit Unsicherheitsband

**Matplotlib (statisch)**

In [ ]:
profil = df.groupby(df.index.hour)["last_kw"].agg(["mean", "std"])

plt.plot(profil.index, profil["mean"], label="Mittlere Last")
plt.fill_between(profil.index,
                 profil["mean"] - profil["std"],
                 profil["mean"] + profil["std"],
                 alpha=0.2, label="±1σ")
plt.title("Durchschnittliches Tagesprofil 2024")
plt.xlabel("Stunde"); plt.ylabel("Last (kW)")
plt.ylim(bottom=0)
plt.xlim(0, 23)
plt.legend()
plt.show()


**Plotly (interaktiv)**

In [ ]:
profil = df.groupby(df.index.hour)["last_kw"].agg(["mean", "std"])

fig = go.Figure([
    go.Scatter(x=profil.index, y=profil["mean"] + profil["std"],
               line=dict(width=0), showlegend=False),
    go.Scatter(x=profil.index, y=profil["mean"] - profil["std"],
               line=dict(width=0), fill="tonexty", name="±1σ"),
    go.Scatter(x=profil.index, y=profil["mean"], name="Mittlere Last"),
])
fig.update_layout(title="Durchschnittliches Tagesprofil 2024",
                  xaxis_title="Stunde", yaxis_title="Last (kW)")
fig.update_yaxes(rangemode="tozero")
fig.show()


## 4.11 Übungsaufgaben

Visualisiere die echten 15-min-Daten aus [../3_Zeitreihen/](../3_Zeitreihen/) —
einmal mit Matplotlib, einmal mit Plotly, plus eine kurze Interpretation.

- `dayahead_2025.csv` — Day-Ahead-Preise DE 2025
- `CHR11_last_15min_2025.csv` — Lastprofil Studentin


In [ ]:
df_da = pd.read_csv("../3_Zeitreihen/dayahead_2025.csv",
                     parse_dates=["timestamp"], index_col="timestamp")
df_ll = pd.read_csv("../3_Zeitreihen/CHR11_last_15min_2025.csv",
                     parse_dates=["timestamp"], index_col="timestamp")

### Aufgabe 1: Preisverlauf 2025

Plotte den Verlauf der Day-Ahead-Preise 2025 und hebe negative Preise farblich hervor.

**Interpretation:** Was fällt am Preisverlauf auf? (1–2 Sätze)


**Matplotlib**

In [ ]:
preis = df_da["dayahead €/MWh"]

preis.plot(title="Day-Ahead-Preise DE 2025", ylabel="Preis (€/MWh)",
           linewidth=0.5)
plt.fill_between(preis.index, preis, 0, where=preis < 0,
                 color="red", alpha=0.4, label="negative Preise")
plt.axhline(0, color="black", linewidth=0.8)        # 0-€-Linie als Referenz
plt.scatter([preis.idxmax(), preis.idxmin()],
            [preis.max(),    preis.min()], color="black", zorder=5)
plt.legend()
plt.show()


**Plotly**

In [ ]:
preis = df_da["dayahead €/MWh"]

fig = px.line(preis, title="Day-Ahead-Preise DE 2025",
              labels={"value": "Preis (€/MWh)", "timestamp": ""})
fig.add_trace(go.Scatter(x=preis.index, y=preis.where(preis < 0),
                         fill="tozeroy", line=dict(width=0),
                         name="negative Preise"))
fig.add_hline(y=0, line_color="black", line_width=1)   # 0-€-Linie
fig.show()


**Interpretation:** Im April 2025 sieht man am selben Tag sowohl Höchst-
als auch Negativrekord — typisch für einen volatilen Spotmarkt mit hohem
EE-Anteil. Negative Preise treten vor allem mittags auf, wenn PV/Wind viel
einspeisen, aber die Last (oder Speicher) nicht mithält.

### Aufgabe 2: Lastprofil-Heatmap

Erstelle eine Heatmap des CHR11-Lastprofils (Tage × Stunden, gemittelt über
die vier 15-min-Werte pro Stunde).

**Interpretation:** Was sagt die Heatmap über den Tagesablauf? (1–2 Sätze)


**Matplotlib**

In [ ]:
last = df_ll["last [kW]"]
heatmap = last.groupby([last.index.date, last.index.hour]).mean().unstack()

plt.figure(figsize=(13, 6))
plt.imshow(heatmap.values, aspect="auto", cmap="YlOrRd")
plt.colorbar(label="Last (kW)")
plt.title("Heatmap CHR11 — Last 2025")
plt.xlabel("Stunde (UTC)"); plt.ylabel("Tag des Jahres")
plt.show()

**Plotly**

In [ ]:
last = df_ll["last [kW]"]
heatmap = last.groupby([last.index.date, last.index.hour]).mean().unstack()

fig = px.imshow(heatmap, aspect="auto", color_continuous_scale="YlOrRd",
                title="Heatmap CHR11 — Last 2025",
                labels=dict(x="Stunde (UTC)", y="Tag", color="kW"))
fig.show()

**Interpretation:** CHR11 hat eine konstant niedrige Grundlast (~50–80 W
Standby) mit Aktivität am späten Vormittag und Abend — kein klassischer
9–17-Uhr-Tag. Die Spitzenlast (3,23 kW) ist ~32× höher als die mittlere Last
(0,1 kW), die Anschlussleistung muss also auf seltene Peaks dimensioniert sein.

### Aufgabe 3: Fixtarif vs. dynamischer Tarif

Vergleiche die monatlichen Stromkosten als gruppiertes Balkendiagramm.
Aufschlag 20 ct/kWh, Energie pro 15-min-Intervall: `last_kw * 0.25`.

**Interpretation:** Welcher Tarif ist günstiger und wann lohnt sich der
dynamische? (1–2 Sätze)


**Matplotlib**

In [ ]:
daten = df_ll.join(df_da, how="inner")
daten.columns = ["last_kw", "boerse_eur_mwh"]
daten["energie_kwh"]    = daten["last_kw"] * 0.25
daten["preis_ct_kwh"]   = daten["boerse_eur_mwh"] / 10 + 20  # + 20 ct/kWh
daten["kosten_dyn"]     = daten["energie_kwh"] * daten["preis_ct_kwh"] / 100
daten["kosten_fix"]     = daten["energie_kwh"] * daten["preis_ct_kwh"].mean() / 100

mk = daten[["kosten_fix", "kosten_dyn"]].resample("ME").sum()
mk = mk[mk.index.year == 2025]            # erste Zeile liegt in 2024 UTC
mk.index = monate[:len(mk)]

mk.plot(kind="bar", rot=0,
        title="Monatlicher Kostenvergleich CHR11 2025",
        ylabel="Kosten (€)")
plt.ylim(bottom=0)
plt.legend(["Fixtarif", "Dynamisch"])
plt.show()


**Plotly**

In [ ]:
fig = px.bar(mk, barmode="group",
             title="Monatlicher Kostenvergleich CHR11 2025",
             labels={"value": "Kosten (€)", "index": "Monat"})
fig.update_yaxes(rangemode="tozero")
fig.show()


**Interpretation:** Der dynamische Tarif ist für CHR11 ~3,5 % teurer,
weil sie tendenziell zu teuren Abendstunden verbraucht (Last↔Preis positiv
korreliert). Lohnen würde er sich erst mit aktiver Lastverschiebung —
PV-Eigenverbrauch, E-Auto-Laden nachts, Wärmepumpe mit Pufferspeicher.